# ETL & Data Wrangling — Advanced Reference
> **Level:** Advanced | **Goal:** Production-grade data cleaning, transformation, and validation pipelines

## Table of Contents
1. [Data Loading & Inspection](#loading)
2. [Missing Data Strategies](#missing)
3. [Outlier Detection & Treatment](#outliers)
4. [String Cleaning & Parsing](#strings)
5. [Date/Time Engineering](#datetime)
6. [Feature Engineering Patterns](#features)
7. [Data Validation with Pandera](#validation)
8. [ETL Pipeline Pattern](#pipeline)
9. [Performance: Large Files with Chunking & Polars](#performance)

In [ ]:
import numpy as np
import pandas as pd
import re
from typing import Optional
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.3f}'.format)

print("Pandas", pd.__version__)

---
## 1 · Data Loading & Inspection <a id='loading'></a>

In [ ]:
# ── Build messy sample dataset ─────────────────────────────────
n = 2000
df_raw = pd.DataFrame({
    'user_id':      rng.integers(1, 500, n),
    'Name':         rng.choice(['Alice Smith','bob jones','CAROL  WHITE ','  David Lee','Eve'], n),
    'email':        rng.choice(['alice@email.com','bob@EMAIL.COM','bad-email','','carol@x.org'], n),
    'Age':          rng.integers(16, 80, n).astype(float),
    'income':       rng.lognormal(10, 0.9, n),
    'gender':       rng.choice(['M','F','male','Female','m','f','Other',None], n),
    'signup_date':  rng.choice(['2023-01-15','01/20/2023','2023-03-32',
                                'March 5, 2023','2022-12-01',np.nan], n),
    'purchase_amt': np.where(rng.random(n) < 0.15, np.nan,
                             rng.lognormal(4, 1, n)),
    'category':     rng.choice(['Electronics','ELECTRONICS','electronics ','Clothing','Food'], n),
    'rating':       rng.choice([1,2,3,4,5,None,99,-1], n),
})

# Inject extra missing values
for col, rate in [('Age', 0.06), ('income', 0.04)]:
    df_raw.loc[rng.random(n) < rate, col] = np.nan

print(f"Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
# ── Profiling function ─────────────────────────────────────────
def profile(df: pd.DataFrame) -> pd.DataFrame:
    report = []
    for col in df.columns:
        s = df[col]
        info = {
            'column':       col,
            'dtype':        str(s.dtype),
            'n_missing':    s.isna().sum(),
            'pct_missing':  f"{s.isna().mean():.1%}",
            'n_unique':     s.nunique(dropna=True),
            'cardinality':  'high' if s.nunique() > 50 else 'low',
        }
        if s.dtype in ['float64','int64','int32']:
            info.update({'mean': s.mean(), 'std': s.std(),
                         'min': s.min(), 'max': s.max()})
        else:
            info.update({'top_value': s.mode()[0] if not s.mode().empty else None})
        report.append(info)
    return pd.DataFrame(report).set_index('column')

profile(df_raw)

---
## 2 · Missing Data Strategies <a id='missing'></a>

| Strategy | When to use |
|---|---|
| Drop rows | < 1–5% missing, row is corrupted |
| Drop columns | > 40–60% missing, not informative |
| Mean/Median imputation | MCAR, numeric, no time dependency |
| Mode imputation | Categorical |
| KNN imputation | Correlated features |
| Model-based (IterativeImputer) | Complex patterns |
| Forward/Backward fill | Time series |
| Missing indicator | When missingness is informative |

In [ ]:
df = df_raw.copy()

# ── Missing indicator + imputation ────────────────────────────
# Preserve missingness signal as a feature
df['income_was_missing']       = df['income'].isna().astype(int)
df['purchase_amt_was_missing'] = df['purchase_amt'].isna().astype(int)
df['age_was_missing']          = df['Age'].isna().astype(int)

# Impute numerics
df['income']       = df['income'].fillna(df['income'].median())
df['purchase_amt'] = df['purchase_amt'].fillna(0)
df['Age']          = df['Age'].fillna(df['Age'].median())

print("Missing after imputation:")
print(df[['income','purchase_amt','Age']].isna().sum())

---
## 3 · Outlier Detection & Treatment <a id='outliers'></a>

In [ ]:
def detect_outliers(series: pd.Series, method: str = 'iqr') -> pd.Series:
    """Return boolean mask: True = outlier."""
    if method == 'iqr':
        q1, q3 = series.quantile([0.25, 0.75])
        iqr = q3 - q1
        return (series < q1 - 1.5*iqr) | (series > q3 + 1.5*iqr)
    elif method == 'zscore':
        z = (series - series.mean()) / series.std()
        return z.abs() > 3
    elif method == 'modified_zscore':
        median = series.median()
        mad = (series - median).abs().median()
        mod_z = 0.6745 * (series - median) / mad
        return mod_z.abs() > 3.5

for col in ['income', 'purchase_amt']:
    mask = detect_outliers(df[col], 'iqr')
    pct  = mask.mean()
    print(f"{col}: {mask.sum()} outliers ({pct:.1%})")

# ── Treatment options ─────────────────────────────────────────
def winsorize(series: pd.Series, lower: float = 0.01, upper: float = 0.99) -> pd.Series:
    """Clip to [p1, p99] percentile."""
    lo, hi = series.quantile([lower, upper])
    return series.clip(lo, hi)

df['income_winsorized'] = winsorize(df['income'])

# Log transform (for right-skewed distributions)
df['income_log'] = np.log1p(df['income'])
print("\nSkewness before:", df['income'].skew().round(2),
      "| after log:",    df['income_log'].skew().round(2))

---
## 4 · String Cleaning & Parsing <a id='strings'></a>

In [ ]:
# ── Name standardization ───────────────────────────────────────
df['name_clean'] = (
    df['Name']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)   # collapse spaces
    .str.title()                              # Title Case
)

# ── Email validation and normalization ─────────────────────────
EMAIL_REGEX = re.compile(r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$')

df['email_clean'] = df['email'].str.lower().str.strip()
df['email_valid'] = df['email_clean'].apply(
    lambda e: bool(EMAIL_REGEX.match(e)) if isinstance(e, str) and e else False
)
df['email_domain'] = df['email_clean'].where(df['email_valid']).str.extract(r'@(.+)')

# ── Gender harmonization ───────────────────────────────────────
gender_map = {
    'm': 'Male', 'male': 'Male', 'f': 'Female',
    'female': 'Female', 'other': 'Other'
}
df['gender_clean'] = (df['gender']
                      .str.lower()
                      .str.strip()
                      .map(gender_map)
                      .fillna('Unknown'))

# ── Category standardization ───────────────────────────────────
df['category_clean'] = (df['category']
                        .str.strip()
                        .str.title()
                        .replace({'Electronics': 'Electronics'}))

print(df[['Name','name_clean','email','email_valid','gender','gender_clean']].head(10))

---
## 5 · Date/Time Engineering <a id='datetime'></a>

In [ ]:
# ── Parse multiple date formats robustly ──────────────────────
def parse_date_robust(date_str) -> Optional[pd.Timestamp]:
    """Try multiple formats; return NaT on failure."""
    if pd.isna(date_str):
        return pd.NaT
    formats = [
        '%Y-%m-%d', '%m/%d/%Y', '%d/%m/%Y',
        '%B %d, %Y', '%b %d, %Y'
    ]
    for fmt in formats:
        try:
            dt = pd.to_datetime(date_str, format=fmt)
            # Sanity check: year must be reasonable
            if 2000 <= dt.year <= 2030:
                return dt
        except (ValueError, TypeError):
            continue
    return pd.NaT

df['signup_date_parsed'] = df['signup_date'].apply(parse_date_robust)
parse_success = df['signup_date_parsed'].notna().mean()
print(f"Date parse success rate: {parse_success:.1%}")

# ── Date feature extraction ────────────────────────────────────
sd = df['signup_date_parsed']
reference_date = pd.Timestamp('2024-01-01')

df['signup_year']     = sd.dt.year
df['signup_month']    = sd.dt.month
df['signup_dow']      = sd.dt.dayofweek   # 0=Mon, 6=Sun
df['signup_is_weekend'] = (sd.dt.dayofweek >= 5).astype(int)
df['signup_quarter']  = sd.dt.quarter
df['tenure_days']     = (reference_date - sd).dt.days

# ── Cyclical encoding (for models that can't handle periodicity) 
df['month_sin'] = np.sin(2 * np.pi * df['signup_month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['signup_month'] / 12)
df['dow_sin']   = np.sin(2 * np.pi * df['signup_dow'] / 7)
df['dow_cos']   = np.cos(2 * np.pi * df['signup_dow'] / 7)

print(df[['signup_date','signup_date_parsed','tenure_days','month_sin','month_cos']].head(5))

---
## 6 · Feature Engineering Patterns <a id='features'></a>

In [ ]:
# ── Binning / Bucketing ────────────────────────────────────────
df['age_group'] = pd.cut(
    df['Age'],
    bins=[0, 25, 35, 50, 65, 100],
    labels=['18-25','26-35','36-50','51-65','65+'],
    right=True
)

df['income_quartile'] = pd.qcut(
    df['income'],
    q=4,
    labels=['Q1','Q2','Q3','Q4']
)

# ── Target encoding (mean encode) ─────────────────────────────
# ⚠️ Always compute on training data only and apply to test!
target = rng.binomial(1, 0.12, len(df))
global_mean = target.mean()
smoothing_factor = 20  # regularization

cat_stats = pd.DataFrame({'target': target, 'category': df['category_clean']})
cat_mean  = cat_stats.groupby('category')['target'].agg(['mean','count'])
cat_mean['smoothed_mean'] = (
    (cat_mean['mean'] * cat_mean['count'] + global_mean * smoothing_factor)
    / (cat_mean['count'] + smoothing_factor)
)
df['category_target_enc'] = df['category_clean'].map(cat_mean['smoothed_mean'])

print(cat_mean.round(4))
print("\nSmoothed target encoding applied.")

In [ ]:
# ── Aggregation features (entity-level features) ──────────────
# Per-user statistics from transaction-level data
user_stats = df.groupby('user_id').agg(
    n_transactions  = ('purchase_amt', 'count'),
    total_spend     = ('purchase_amt', 'sum'),
    avg_spend       = ('purchase_amt', 'mean'),
    std_spend       = ('purchase_amt', 'std'),
    max_spend       = ('purchase_amt', 'max'),
    n_categories    = ('category_clean', 'nunique'),
).reset_index()

user_stats['cv_spend']     = user_stats['std_spend'] / (user_stats['avg_spend'] + 1)
user_stats['spend_per_cat'] = user_stats['total_spend'] / user_stats['n_categories']

# Merge back to main dataframe
df = df.merge(user_stats, on='user_id', how='left', suffixes=('', '_user'))
print(f"Added {len(user_stats.columns)-1} user-level features")
user_stats.head()

---
## 7 · Data Validation with Pandera <a id='validation'></a>

In [ ]:
try:
    import pandera as pa
    from pandera import Column, DataFrameSchema, Check

    schema = DataFrameSchema({
        'Age': Column(
            float,
            checks=[
                Check.greater_than_or_equal_to(0),
                Check.less_than_or_equal_to(120),
            ],
            nullable=True
        ),
        'income': Column(
            float,
            checks=Check.greater_than(0),
            nullable=False
        ),
        'purchase_amt': Column(
            float,
            checks=Check.greater_than_or_equal_to(0),
            nullable=True
        ),
        'email_valid': Column(int, checks=Check.isin([0, 1])),
    })

    validated = schema.validate(df[['Age','income','purchase_amt','email_valid']])
    print(f"Validation passed. {len(validated)} rows validated.")

except ImportError:
    print("Install: pip install pandera")
    print("\nManual validation instead:")

    issues = []
    if (df['Age'] < 0).any():
        issues.append("Age has negative values")
    if (df['income'] <= 0).any():
        issues.append("Income has non-positive values")
    if not df['email_valid'].isin([0, 1]).all():
        issues.append("email_valid has values outside {0,1}")

    if issues:
        for i in issues:
            print(f"  ISSUE: {i}")
    else:
        print("  All manual checks passed.")

---
## 8 · ETL Pipeline Pattern <a id='pipeline'></a>

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, List
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

@dataclass
class TransformStep:
    name: str
    func: Callable[[pd.DataFrame], pd.DataFrame]

class ETLPipeline:
    """
    Composable ETL pipeline with logging and shape tracking.
    Each step receives and returns a DataFrame.
    """
    def __init__(self, steps: List[TransformStep]):
        self.steps = steps

    def run(self, df: pd.DataFrame) -> pd.DataFrame:
        logger.info(f"Starting pipeline. Input: {df.shape}")
        for step in self.steps:
            try:
                df = step.func(df)
                logger.info(f"  [{step.name}] OK → {df.shape}")
            except Exception as e:
                logger.error(f"  [{step.name}] FAILED: {e}")
                raise
        logger.info(f"Pipeline complete. Output: {df.shape}")
        return df

# ── Define steps ───────────────────────────────────────────────
def step_deduplicate(df):
    return df.drop_duplicates(subset=['user_id','signup_date'])

def step_standardize_strings(df):
    df = df.copy()
    df['name_clean'] = df['Name'].str.strip().str.title()
    df['category_clean'] = df['category'].str.strip().str.title()
    return df

def step_handle_missing(df):
    df = df.copy()
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['purchase_amt'] = df['purchase_amt'].fillna(0)
    return df

def step_validate_rating(df):
    df = df.copy()
    df['rating'] = df['rating'].where(df['rating'].between(1, 5))
    return df

pipeline = ETLPipeline([
    TransformStep('Deduplicate',       step_deduplicate),
    TransformStep('Standardize strings', step_standardize_strings),
    TransformStep('Handle missing',    step_handle_missing),
    TransformStep('Validate rating',   step_validate_rating),
])

df_clean = pipeline.run(df_raw)

---
## 9 · Performance: Large Files with Chunking & Polars <a id='performance'></a>

In [ ]:
import io

# ── Chunked reading for large CSV files ───────────────────────
# When file doesn't fit in memory: read + aggregate in chunks

# Simulate a large CSV
large_df = pd.DataFrame({
    'user_id': rng.integers(1, 1000, 100_000),
    'revenue': rng.lognormal(4, 1, 100_000),
    'category': rng.choice(['A','B','C'], 100_000)
})
buf = io.StringIO()
large_df.to_csv(buf, index=False)
buf.seek(0)

# Process in chunks
results = []
for chunk in pd.read_csv(buf, chunksize=10_000):
    chunk_summary = chunk.groupby('category')['revenue'].agg(['sum','count'])
    results.append(chunk_summary)

final = pd.concat(results).groupby(level=0).sum()
final['avg_revenue'] = final['sum'] / final['count']
print("Chunked aggregation result:")
print(final.round(2))

In [ ]:
# ── Polars: fast DataFrame library (Rust-based) ────────────────
try:
    import polars as pl
    import time

    n = 5_000_000
    data = {
        'user_id':  rng.integers(1, 10000, n).tolist(),
        'revenue':  rng.lognormal(4, 1, n).tolist(),
        'category': rng.choice(['A','B','C','D'], n).tolist()
    }

    # Pandas
    t0 = time.perf_counter()
    pd_df = pd.DataFrame(data)
    pd_result = pd_df.groupby('category')['revenue'].mean()
    pd_time = time.perf_counter() - t0

    # Polars
    t0 = time.perf_counter()
    pl_df = pl.DataFrame(data)
    pl_result = pl_df.group_by('category').agg(pl.col('revenue').mean())
    pl_time = time.perf_counter() - t0

    print(f"pandas: {pd_time:.3f}s | polars: {pl_time:.3f}s  ({pd_time/pl_time:.1f}x speedup)")

    # Polars lazy API (query optimization)
    lazy_result = (
        pl.LazyFrame(data)
        .filter(pl.col('revenue') > 20)
        .group_by('category')
        .agg([
            pl.col('revenue').mean().alias('avg_revenue'),
            pl.col('revenue').count().alias('n_transactions')
        ])
        .sort('avg_revenue', descending=True)
        .collect()   # executes optimized query plan
    )
    print(lazy_result)

except ImportError:
    print("Install: pip install polars")
    print("Polars advantages:")
    print("  - 5-20x faster than pandas for aggregations")
    print("  - Lazy evaluation (query optimizer)")
    print("  - Parallel execution out of the box")
    print("  - Memory efficient (Apache Arrow format)")